# Lab 3: Conditional Edges
Conditional edges allow you to route execution dynamically based on the current state contents or an LLM's output. You specify:
1. The source node.
2. A routing function that returns the next node's name.
3. A mapping of string names to target nodes.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### Define State, Nodes, Router function and compile

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class RouterState(TypedDict):
    query: str
    is_question: bool
    reply: str

def classifier_node(state: RouterState):
    print("--- Classifying Input ---")
    is_q = state["query"].strip().endswith("?")
    return {"is_question": is_q}

def question_handler(state: RouterState):
    print("--- Handling Question ---")
    return {"reply": "This is an answer to your question!"}

def statement_handler(state: RouterState):
    print("--- Handling Statement ---")
    return {"reply": "Understood. Thank you for sharing."}

# Routing Function
def route_decision(state: RouterState):
    if state["is_question"]:
        return "question_path"
    else:
        return "statement_path"

# Build Graph
router_builder = StateGraph(RouterState)
router_builder.add_node("classifier", classifier_node)
router_builder.add_node("question", question_handler)
router_builder.add_node("statement", statement_handler)

router_builder.add_edge(START, "classifier")

# Add conditional edge from classifier node
router_builder.add_conditional_edges(
    "classifier",
    route_decision,
    {
        "question_path": "question",
        "statement_path": "statement"
    }
)

router_builder.add_edge("question", END)
router_builder.add_edge("statement", END)

router_graph = router_builder.compile()

### Test Routing Decisions

In [3]:
print("--- Scenario A ---")
print(router_graph.invoke({"query": "What is the capital of France?", "is_question": False, "reply": ""}))

print("\n--- Scenario B ---")
print(router_graph.invoke({"query": "The sun rises in the east.", "is_question": False, "reply": ""}))

--- Scenario A ---
--- Classifying Input ---
--- Handling Question ---
{'query': 'What is the capital of France?', 'is_question': True, 'reply': 'This is an answer to your question!'}

--- Scenario B ---
--- Classifying Input ---
--- Handling Statement ---
{'query': 'The sun rises in the east.', 'is_question': False, 'reply': 'Understood. Thank you for sharing.'}
